# LD-VQN on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdmAnders/LD-VQN-Reid/blob/main/notebooks/LD-VQN_Colab.ipynb)

Official code of *"A large-scale multimodal benchmark and linguistically driven network for vehicle re-identification"*
(Ergin & Keçeli, Neurocomputing 2027) – [repository](https://github.com/AdmAnders/LD-VQN-Reid).

This notebook installs the code, prepares one dataset (**VeRi-776, VehicleID or VeRi-Wild**) and lets you

1. run a **10-minute pipeline test** on real data,
2. **evaluate** the released weights,
3. **train** LD-VQN (auto-resumes after a Colab disconnect),
4. draw the **attention maps** (Fig. 9) and the **t-SNE** plot (Fig. 11).

**Before you start**
- *Runtime → Change runtime type → GPU.* The paper setting (CLIP ViT-L/14-336, batch 96, fp32) needs a large GPU
  (A100 80 GB / H100). On smaller GPUs set `LOW_MEMORY = True` below (mixed precision + smaller batch; results may
  differ slightly). Evaluation runs on any GPU.
- The vehicle **images are not distributed** with this repository. Request VeRi-776, VehicleID or VERI-Wild from their
  owners and put the archive in your Google Drive. The captions come with the repository.

## 1. GPU and code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, shlex
REPO_URL = "https://github.com/AdmAnders/LD-VQN-Reid.git"
REPO_DIR = "/content/LD-VQN-Reid"

if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --format="commit %h  %cd  %s" --date=short
# Colab's own torch/torchvision are kept; only the model/tokenizer stack is pinned.
!pip install -q -r requirements-colab.txt
!python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__)"

## 2. Google Drive and settings

Edit the cell below, then run it. Checkpoints are written to Drive (`OUTPUT_DIR`), so training continues where it
stopped if the Colab session ends: just run sections 1, 2 and 6 again.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DATASET = "veri776"            # "veri776" | "vehicleid" | "veriwild"

# Image archive on your Drive (.zip) or an already extracted folder on Drive.
IMAGES_SRC = {
    "veri776":   "/content/drive/MyDrive/Dataset/VeRi.zip",        # -> VeRi/image_train, image_query, image_test
    "vehicleid": "/content/drive/MyDrive/Dataset/VehicleID.zip",   # -> VehicleID/image, VehicleID/train_test_split
    "veriwild":  "/content/drive/MyDrive/Dataset/VeRi-Wild.zip",   # -> VeRi-Wild/images/<id>/<img>.jpg
}[DATASET]

DATA_DIR = "/content/data"     # images are copied to the local disk (much faster than reading from Drive)
# Folder that contains the split folders after extraction - adjust if your archive uses another top folder name.
IMAGE_ROOT = {"veri776": f"{DATA_DIR}/VeRi",
              "vehicleid": f"{DATA_DIR}/VehicleID",
              "veriwild": f"{DATA_DIR}/VeRi-Wild"}[DATASET]
SPLIT_DIR = f"{IMAGE_ROOT}/train_test_split"     # VehicleID only: original train_list.txt / test_list_*.txt

# Optional: Drive folder with the FULL caption files (the VehicleID / VeRi-Wild files in the repo are demo subsets).
FULL_CAPTIONS_DIR = ""

TEST_SIZE = {"veri776": None, "vehicleid": 800, "veriwild": 3000}[DATASET]   # split used during training
OUTPUT_DIR = f"/content/drive/MyDrive/LD-VQN/outputs/{DATASET}"
WEIGHTS = f"/content/drive/MyDrive/LD-VQN/weights/ldvqn_{DATASET}.pth"        # released weights (section 4)
LOW_MEMORY = False             # True: mixed precision + batch 48 x 8 accumulation (for GPUs < 80 GB)
RERANK = True                  # k-reciprocal re-ranking (set False for VeRi-Wild 5000/10000: needs a lot of RAM)

CONFIG = f"configs/{DATASET}_ldvqn.yml"

def opts(*extra):
    # config overrides passed to train.py / test.py / tools
    o = ["DATA.IMAGE_ROOT", IMAGE_ROOT, "EXPERIMENT.OUTPUT_DIR", OUTPUT_DIR]
    if DATASET == "vehicleid":
        o += ["DATA.SPLIT_DIR", SPLIT_DIR]
    if TEST_SIZE:
        o += ["DATA.TEST_SIZE", str(TEST_SIZE)]
    if FULL_CAPTIONS_DIR:
        o += ["DATA.CAPTION_DIR", FULL_CAPTIONS_DIR]
    if LOW_MEMORY:
        o += ["SOLVER.AMP", "true", "DATALOADER.BATCH_SIZE", "48", "SOLVER.ACCUMULATION_STEPS", "8"]
    if not RERANK:
        o += ["TEST.RERANKING", "false"]
    o += [str(x) for x in extra]
    return " ".join(shlex.quote(x) for x in o)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("config :", CONFIG)
print("opts   :", opts())

## 3. Prepare the data and check the setup

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
if os.path.isdir(IMAGE_ROOT) and os.listdir(IMAGE_ROOT):
    print("Images already present:", IMAGE_ROOT)
elif IMAGES_SRC.endswith(".zip"):
    print("Extracting", IMAGES_SRC, "->", DATA_DIR, "(a few minutes)")
    SRC = shlex.quote(IMAGES_SRC)
    !unzip -q -o {SRC} -d {DATA_DIR}
elif os.path.isdir(IMAGES_SRC):
    SRC, DST = shlex.quote(IMAGES_SRC), shlex.quote(IMAGE_ROOT)
    !cp -r {SRC} {DST}
else:
    raise FileNotFoundError(f"IMAGES_SRC not found: {IMAGES_SRC}")
!ls {DATA_DIR}
ROOT = shlex.quote(IMAGE_ROOT)
!ls {ROOT} | head

In [ ]:
# Counts captions / identities / missing images per split and gives a GPU-memory hint.
OPTS = opts()
!python tools/check_setup.py --config {CONFIG} --opts {OPTS}

## 4. Quick pipeline test (~10 min)

Trains 20 iterations on the real data and evaluates once without re-ranking. Output goes to `/content/smoke_test`
(not Drive). The numbers are meaningless; the test only shows that every stage runs.

In [ ]:
SMOKE_DIR = f"/content/smoke_test/{DATASET}"
!rm -rf {SMOKE_DIR}
smoke_opts = opts("EXPERIMENT.OUTPUT_DIR", SMOKE_DIR, "SOLVER.EPOCHS", 1, "SOLVER.MAX_ITERS_PER_EPOCH", 20,
                  "TEST.EVAL_PERIOD", 1, "TEST.RERANKING", "false")
!python -u train.py --config {CONFIG} --opts {smoke_opts}

## 5. Evaluate the released weights

Downloads the checkpoint of the selected dataset from the GitHub release (or uses `WEIGHTS` if it is already on
Drive). With the VeRi-776 checkpoint this should print **mAP 78.26 % / Rank-1 98.39 %**.

In [ ]:
import urllib.request
WEIGHTS_URL = f"https://github.com/AdmAnders/LD-VQN-Reid/releases/download/v1.0/ldvqn_{DATASET}.pth"
if not os.path.isfile(WEIGHTS):
    os.makedirs(os.path.dirname(WEIGHTS), exist_ok=True)
    print("Downloading", WEIGHTS_URL)
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS)
W = shlex.quote(WEIGHTS)
!sha256sum {W}

TEST_SIZES = {"veri776": "", "vehicleid": "--test_sizes 800 1600 2400", "veriwild": "--test_sizes 3000 5000 10000"}[DATASET]
RESULTS = shlex.quote(OUTPUT_DIR + "/test_results.json")
OPTS = opts()
!python -u test.py --config {CONFIG} --weights {W} {TEST_SIZES} --output {RESULTS} --opts {OPTS}

## 6. Train

Runs the full schedule of the config (60 epochs, evaluation every 5 epochs, best model → `OUTPUT_DIR/ldvqn_best.pth`).
A full VeRi-776 run takes about a day on a large GPU, longer than one Colab session: when the session ends, run
sections 1, 2 and this cell again – training resumes from `OUTPUT_DIR/ldvqn_checkpoint.pth`.
The log is appended to `OUTPUT_DIR/train.log`.

In [ ]:
OPTS, LOG = opts(), shlex.quote(OUTPUT_DIR + "/train.log")
!python -u train.py --config {CONFIG} --opts {OPTS} 2>&1 | tee -a {LOG}

## 7. Visualizations (Fig. 9 and Fig. 11)

In [ ]:
import sys
from IPython.display import Image, display
sys.path.insert(0, REPO_DIR)
from utils import load_config
from datasets import build_records

best = f"{OUTPUT_DIR}/ldvqn_best.pth"
VIS_WEIGHTS = best if os.path.isfile(best) else WEIGHTS
cfg = load_config(CONFIG, shlex.split(opts()))
_, query_records, _ = build_records(cfg)
sample = query_records[0]
print(sample["img_path"], "\n", sample["caption"])

out = f"/content/vis/{DATASET}_attention.png"
VW, IMG, CAP, OPTS = shlex.quote(VIS_WEIGHTS), shlex.quote(sample["img_path"]), shlex.quote(sample["caption"] or "A vehicle."), opts()
!python tools/visualize_attention.py --config {CONFIG} --weights {VW} --image {IMG} --caption {CAP} --output {out} --opts {OPTS}
display(Image(out, width=700))

In [ ]:
out = f"/content/vis/{DATASET}_tsne.png"
VW, OPTS = shlex.quote(VIS_WEIGHTS), opts()
!python tools/tsne_visualization.py --config {CONFIG} --weights {VW} --num_ids 10 --output {out} --opts {OPTS}
display(Image(out, width=600))